# Day 4 Assignment 3 — IoT Sensor Data Analysis

**Dataset:** `iot_sensor_data_raw.csv`

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("iot_sensor_data_raw.csv")
display(df.head())

,Timestamp,Device_ID,Temperature,Humidity,Pressure,Vibration,Battery_Level,Location,Machine_Status
0,2026-04-21 22:30:00,DEV_005,65.89,62.70,1007.96,2.22,46.51,Factory_A,Warning
1,2026-01-22 06:15:00,DEV_007,70.65,70.95,1021.85,2.73,57.04,Factory_A,Normal
2,2026-04-01 07:00:00,DEV_005,81.01,60.49,1009.82,2.15,11.65,Factory_A,Normal
3,2026-01-12 14:30:00,DEV_007,74.17,68.76,1001.42,2.03,17.36,Factory_C,Critical
4,2026-05-25 19:30:00,DEV_008,74.04,54.32,1014.34,NaN,27.04,Factory_C,Normal


## Q1. Load the dataset.

**Answer:**

In [2]:
df = pd.read_csv("iot_sensor_data_raw.csv")
print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
display(df.head())

Dataset loaded successfully.
Rows: 20000
Columns: 9


,Timestamp,Device_ID,Temperature,Humidity,Pressure,Vibration,Battery_Level,Location,Machine_Status
0,2026-04-21 22:30:00,DEV_005,65.89,62.70,1007.96,2.22,46.51,Factory_A,Warning
1,2026-01-22 06:15:00,DEV_007,70.65,70.95,1021.85,2.73,57.04,Factory_A,Normal
2,2026-04-01 07:00:00,DEV_005,81.01,60.49,1009.82,2.15,11.65,Factory_A,Normal
3,2026-01-12 14:30:00,DEV_007,74.17,68.76,1001.42,2.03,17.36,Factory_C,Critical
4,2026-05-25 19:30:00,DEV_008,74.04,54.32,1014.34,NaN,27.04,Factory_C,Normal


## Q2. Display its dimensions and structure.

**Answer:**

In [3]:
print("Dimensions:", df.shape)
print("\nStructure:")
df.info()

Dimensions: (20000, 9)

Structure:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Timestamp       20000 non-null  object 
 1   Device_ID       20000 non-null  object 
 2   Temperature     19701 non-null  float64
 3   Humidity        19700 non-null  float64
 4   Pressure        19700 non-null  float64
 5   Vibration       19702 non-null  float64
 6   Battery_Level   20000 non-null  float64
 7   Location        20000 non-null  object 
 8   Machine_Status  20000 non-null  object 
dtypes: float64(5), object(4)
memory usage: 1.4+ MB


## Q3. Convert Timestamp into Pandas datetime.

**Answer:**

In [4]:
df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
print("Timestamp dtype:", df["Timestamp"].dtype)
display(df[["Timestamp"]].head())

Timestamp dtype: datetime64[ns]


,Timestamp
0,2026-04-21 22:30:00
1,2026-01-22 06:15:00
2,2026-04-01 07:00:00
3,2026-01-12 14:30:00
4,2026-05-25 19:30:00


## Q4. Check whether the timestamps are correctly ordered.

**Answer:**

In [5]:
is_ordered = df["Timestamp"].is_monotonic_increasing
print("Timestamps correctly ordered:", is_ordered)
if not is_ordered:
    print("The timestamps are not in chronological order, so the working DataFrame will be sorted by Device_ID and Timestamp for time-series operations.")
    df = df.sort_values(["Device_ID", "Timestamp"]).reset_index(drop=True)
print("After sorting, timestamps within each device are ordered:",
      df.groupby("Device_ID")["Timestamp"].apply(lambda s: s.is_monotonic_increasing).all())

Timestamps correctly ordered: False
The timestamps are not in chronological order, so the working DataFrame will be sorted by Device_ID and Timestamp for time-series operations.
After sorting, timestamps within each device are ordered: True


## Q5. Identify missing sensor readings.

**Answer:**

In [6]:
sensor_cols = ["Temperature", "Humidity", "Pressure", "Vibration"]
missing_counts = df[sensor_cols].isna().sum().rename("Missing_Count")
print("Missing sensor readings:")
display(missing_counts.to_frame())

Missing sensor readings:


,Missing_Count
Temperature,299
Humidity,300
Pressure,300
Vibration,298


## Q6. Calculate the percentage of missing values for each sensor.

**Answer:**

In [7]:
missing_pct = (df[sensor_cols].isna().mean() * 100).round(2).rename("Missing_Percentage")
display(missing_pct.to_frame())

,Missing_Percentage
Temperature,1.50
Humidity,1.50
Pressure,1.50
Vibration,1.49


## Q7. Handle missing sensor values using an appropriate method.

**Answer:**

In [8]:
# Linear interpolation is appropriate for time-series sensor readings.
# It is performed separately for each device after chronological sorting.
for col in sensor_cols:
    df[col] = df.groupby("Device_ID")[col].transform(
        lambda s: s.interpolate(method="linear", limit_direction="both")
    )

print("Missing values after interpolation:")
display(df[sensor_cols].isna().sum().to_frame("Remaining_Missing"))

Missing values after interpolation:


,Remaining_Missing
Temperature,0
Humidity,0
Pressure,0
Vibration,0


## Q8. Identify abnormal temperature readings.

**Answer:**

In [9]:
q1_temp = df["Temperature"].quantile(0.25)
q3_temp = df["Temperature"].quantile(0.75)
iqr_temp = q3_temp - q1_temp
temp_warn_low = q1_temp - 1.5 * iqr_temp
temp_warn_high = q3_temp + 1.5 * iqr_temp
temp_crit_low = q1_temp - 3 * iqr_temp
temp_crit_high = q3_temp + 3 * iqr_temp

df["Temp_Abnormal"] = (df["Temperature"] < temp_warn_low) | (df["Temperature"] > temp_warn_high)

print(f"Temperature IQR thresholds: warning below {temp_warn_low:.3f} or above {temp_warn_high:.3f}")
print(f"Critical temperature thresholds: below {temp_crit_low:.3f} or above {temp_crit_high:.3f}")
print("Number of abnormal temperature readings:", int(df["Temp_Abnormal"].sum()))
display(df.loc[df["Temp_Abnormal"], ["Timestamp","Device_ID","Temperature","Location"]].head(20))

Temperature IQR thresholds: warning below 48.830 or above 91.310
Critical temperature thresholds: below 32.900 or above 107.240
Number of abnormal temperature readings: 196


,Timestamp,Device_ID,Temperature,Location
59,2026-01-05 10:30:00,DEV_001,103.180000,Factory_C
236,2026-01-19 07:00:00,DEV_001,42.940000,Factory_B
508,2026-02-11 10:00:00,DEV_001,120.852014,Factory_A
638,2026-02-21 15:30:00,DEV_001,46.660000,Factory_A
697,2026-02-26 02:45:00,DEV_001,143.801135,Factory_A
873,2026-03-11 02:00:00,DEV_001,91.670000,Factory_C
893,2026-03-13 19:00:00,DEV_001,97.110000,Factory_B
1135,2026-04-05 02:45:00,DEV_001,44.730000,Factory_A
1253,2026-04-13 21:30:00,DEV_001,47.050000,Factory_B
1491,2026-05-05 03:45:00,DEV_001,131.939242,Factory_B


## Q9. Identify abnormal vibration readings.

**Answer:**

In [10]:
q1_vib = df["Vibration"].quantile(0.25)
q3_vib = df["Vibration"].quantile(0.75)
iqr_vib = q3_vib - q1_vib
vib_warn_low = q1_vib - 1.5 * iqr_vib
vib_warn_high = q3_vib + 1.5 * iqr_vib
vib_crit_low = q1_vib - 3 * iqr_vib
vib_crit_high = q3_vib + 3 * iqr_vib

df["Vibration_Abnormal"] = (df["Vibration"] < vib_warn_low) | (df["Vibration"] > vib_warn_high)

print(f"Vibration IQR thresholds: warning below {vib_warn_low:.3f} or above {vib_warn_high:.3f}")
print(f"Critical vibration thresholds: below {vib_crit_low:.3f} or above {vib_crit_high:.3f}")
print("Number of abnormal vibration readings:", int(df["Vibration_Abnormal"].sum()))
display(df.loc[df["Vibration_Abnormal"], ["Timestamp","Device_ID","Vibration","Location"]].head(20))

Vibration IQR thresholds: warning below 0.325 or above 4.685
Critical vibration thresholds: below -1.310 or above 6.320
Number of abnormal vibration readings: 169


,Timestamp,Device_ID,Vibration,Location
120,2026-01-10 07:00:00,DEV_001,4.990000,Factory_A
155,2026-01-12 21:15:00,DEV_001,11.306348,Factory_B
195,2026-01-16 03:30:00,DEV_001,13.881332,Factory_B
245,2026-01-20 06:45:00,DEV_001,0.020000,Factory_C
279,2026-01-22 17:45:00,DEV_001,0.240000,Factory_A
299,2026-01-24 04:45:00,DEV_001,14.639271,Factory_A
447,2026-02-05 19:00:00,DEV_001,4.990000,Factory_A
467,2026-02-07 10:15:00,DEV_001,4.710000,Factory_B
507,2026-02-11 06:30:00,DEV_001,4.810000,Factory_C
517,2026-02-12 05:00:00,DEV_001,4.800000,Factory_B


## Q10. Identify machines with critically low battery levels.

**Answer:**

In [11]:
critical_battery = df[df["Battery_Level"] < 20].copy()
print("Number of critically low battery records:", len(critical_battery))
print("Devices affected:", critical_battery["Device_ID"].nunique())
display(critical_battery[["Timestamp","Device_ID","Battery_Level","Location"]].head(20))

Number of critically low battery records: 2317
Devices affected: 8


,Timestamp,Device_ID,Battery_Level,Location
2,2026-01-01 09:30:00,DEV_001,11.780000,Factory_A
6,2026-01-01 18:00:00,DEV_001,13.920000,Factory_A
12,2026-01-02 10:15:00,DEV_001,15.700000,Factory_C
13,2026-01-02 10:45:00,DEV_001,15.920000,Factory_A
33,2026-01-03 20:30:00,DEV_001,13.950000,Factory_A
34,2026-01-03 21:00:00,DEV_001,10.040000,Factory_A
43,2026-01-04 10:30:00,DEV_001,15.980000,Factory_A
75,2026-01-06 13:15:00,DEV_001,17.100000,Factory_C
76,2026-01-06 16:00:00,DEV_001,16.190000,Factory_A
79,2026-01-06 23:00:00,DEV_001,12.350000,Factory_C


## Q11. Explain how you decided what constitutes an abnormal reading.

**Answer:**

In [12]:
print("Method:")
print("1. Temperature and Vibration are continuous sensor variables, so the IQR outlier method is used.")
print("2. Values outside Q1 - 1.5×IQR or Q3 + 1.5×IQR are treated as abnormal/warning readings.")
print("3. Values outside Q1 - 3×IQR or Q3 + 3×IQR are treated as critical sensor readings.")
print("4. Battery uses the assignment's explicit rule: >=50 Healthy, 20–49 Moderate, <20 Critical.")
print("This approach is data-driven and avoids choosing arbitrary temperature/vibration limits.")

Method:
1. Temperature and Vibration are continuous sensor variables, so the IQR outlier method is used.
2. Values outside Q1 - 1.5×IQR or Q3 + 1.5×IQR are treated as abnormal/warning readings.
3. Values outside Q1 - 3×IQR or Q3 + 3×IQR are treated as critical sensor readings.
4. Battery uses the assignment's explicit rule: >=50 Healthy, 20–49 Moderate, <20 Critical.
This approach is data-driven and avoids choosing arbitrary temperature/vibration limits.


## Q12. Calculate the average temperature by hour.

**Answer:**

In [13]:
df["Hour"] = df["Timestamp"].dt.hour
avg_temp_by_hour = df.groupby("Hour")["Temperature"].mean().round(2)
display(avg_temp_by_hour.to_frame("Average_Temperature"))

,Average_Temperature
Hour,
0,70.10
1,70.53
2,70.01
3,69.71
4,69.83
5,70.28
6,70.26
7,70.67
8,70.45


## Q13. Calculate the average temperature for each device.

**Answer:**

In [14]:
avg_temp_device = df.groupby("Device_ID")["Temperature"].mean().sort_values(ascending=False).round(2)
display(avg_temp_device.to_frame("Average_Temperature"))

,Average_Temperature
Device_ID,
DEV_004,70.59
DEV_002,70.38
DEV_008,70.32
DEV_001,70.26
DEV_005,70.14
DEV_007,70.12
DEV_006,70.01
DEV_003,69.94


## Q14. Calculate the average vibration for each device.

**Answer:**

In [15]:
avg_vibration_device = df.groupby("Device_ID")["Vibration"].mean().sort_values(ascending=False).round(3)
display(avg_vibration_device.to_frame("Average_Vibration"))

,Average_Vibration
Device_ID,
DEV_002,2.568
DEV_006,2.557
DEV_001,2.554
DEV_003,2.540
DEV_008,2.528
DEV_004,2.527
DEV_007,2.523
DEV_005,2.495


## Q15. Find the maximum temperature recorded by each device.

**Answer:**

In [16]:
max_temp_device = df.groupby("Device_ID")["Temperature"].max().sort_values(ascending=False).round(3)
display(max_temp_device.to_frame("Maximum_Temperature"))

,Maximum_Temperature
Device_ID,
DEV_001,159.747
DEV_003,157.604
DEV_007,153.878
DEV_008,153.622
DEV_005,152.974
DEV_004,151.965
DEV_002,151.819
DEV_006,136.003


## Q16. Find the minimum battery level for each device.

**Answer:**

In [17]:
min_battery_device = df.groupby("Device_ID")["Battery_Level"].min().sort_values().round(3)
display(min_battery_device.to_frame("Minimum_Battery_Level"))

,Minimum_Battery_Level
Device_ID,
DEV_008,2.012
DEV_007,2.146
DEV_002,2.155
DEV_006,2.201
DEV_001,2.257
DEV_005,2.957
DEV_003,3.671
DEV_004,8.514


## Q17. Determine which device has the highest average vibration.

**Answer:**

In [18]:
highest_vibration_device = avg_vibration_device.idxmax()
highest_vibration_value = avg_vibration_device.max()
print(f"Device with the highest average vibration: {highest_vibration_device}")
print(f"Average vibration: {highest_vibration_value:.3f}")

Device with the highest average vibration: DEV_002
Average vibration: 2.568


## Q18. Determine which factory has the highest average temperature.

**Answer:**

In [19]:
avg_temp_factory = df.groupby("Location")["Temperature"].mean().sort_values(ascending=False).round(2)
display(avg_temp_factory.to_frame("Average_Temperature"))
print("Factory with the highest average temperature:", avg_temp_factory.idxmax())

,Average_Temperature
Location,
Factory_C,70.29
Factory_A,70.24
Factory_B,70.13


Factory with the highest average temperature: Factory_C


## Q19. Create a Battery_Status column.

**Answer:**

In [20]:
df["Battery_Status"] = np.select(
    [df["Battery_Level"] >= 50, df["Battery_Level"] >= 20],
    ["Healthy", "Moderate"],
    default="Critical"
)
display(df[["Battery_Level","Battery_Status"]].head(10))

,Battery_Level,Battery_Status
0,65.00,Healthy
1,95.38,Healthy
2,11.78,Critical
3,78.19,Healthy
4,63.98,Healthy
5,68.97,Healthy
6,13.92,Critical
7,96.24,Healthy
8,43.17,Moderate
9,87.10,Healthy


## Q20. Create a Temperature_Status column.

**Answer:**

In [21]:
df["Temperature_Status"] = np.select(
    [
        (df["Temperature"] < temp_crit_low) | (df["Temperature"] > temp_crit_high),
        (df["Temperature"] < temp_warn_low) | (df["Temperature"] > temp_warn_high)
    ],
    ["Critical", "Warning"],
    default="Normal"
)
display(df[["Temperature","Temperature_Status"]].head(10))

,Temperature,Temperature_Status
0,68.710,Normal
1,77.100,Normal
2,68.140,Normal
3,76.890,Normal
4,61.220,Normal
5,70.025,Normal
6,78.830,Normal
7,85.480,Normal
8,65.340,Normal
9,68.750,Normal


## Q21. Create a Vibration_Status column.

**Answer:**

In [22]:
df["Vibration_Status"] = np.select(
    [
        (df["Vibration"] < vib_crit_low) | (df["Vibration"] > vib_crit_high),
        (df["Vibration"] < vib_warn_low) | (df["Vibration"] > vib_warn_high)
    ],
    ["Critical", "Warning"],
    default="Normal"
)
display(df[["Vibration","Vibration_Status"]].head(10))

,Vibration,Vibration_Status
0,3.07,Normal
1,2.10,Normal
2,3.04,Normal
3,2.68,Normal
4,2.46,Normal
5,2.10,Normal
6,1.67,Normal
7,1.77,Normal
8,2.00,Normal
9,2.09,Normal


## Q22. Create an overall Machine_Health column.

**Answer:**

In [23]:
df["Machine_Health"] = np.select(
    [
        (df["Battery_Status"] == "Critical") |
        (df["Temperature_Status"] == "Critical") |
        (df["Vibration_Status"] == "Critical"),
        (df["Battery_Status"] == "Moderate") |
        (df["Temperature_Status"] == "Warning") |
        (df["Vibration_Status"] == "Warning")
    ],
    ["Critical", "Warning"],
    default="Normal"
)
display(df[["Device_ID","Battery_Status","Temperature_Status","Vibration_Status","Machine_Health"]].head(15))
print("\nMachine health counts:")
display(df["Machine_Health"].value_counts().to_frame("Count"))

,Device_ID,Battery_Status,Temperature_Status,Vibration_Status,Machine_Health
0,DEV_001,Healthy,Normal,Normal,Normal
1,DEV_001,Healthy,Normal,Normal,Normal
2,DEV_001,Critical,Normal,Normal,Critical
3,DEV_001,Healthy,Normal,Normal,Normal
4,DEV_001,Healthy,Normal,Normal,Normal
5,DEV_001,Healthy,Normal,Normal,Normal
6,DEV_001,Critical,Normal,Normal,Critical
7,DEV_001,Healthy,Normal,Normal,Normal
8,DEV_001,Moderate,Normal,Normal,Warning
9,DEV_001,Healthy,Normal,Normal,Normal



Machine health counts:


,Count
Machine_Health,
Normal,10780
Warning,6814
Critical,2406


## Q23. Find devices that have experienced critical conditions more than five times.

**Answer:**

In [24]:
critical_counts = df.groupby("Device_ID")["Machine_Health"].apply(lambda s: (s == "Critical").sum()).sort_values(ascending=False)
critical_devices = critical_counts[critical_counts > 5]
display(critical_devices.to_frame("Critical_Condition_Count"))
print("All listed devices have experienced critical conditions more than five times." if len(critical_devices) else "No device exceeded five critical conditions.")

,Critical_Condition_Count
Device_ID,
DEV_004,323
DEV_005,318
DEV_002,313
DEV_006,306
DEV_001,304
DEV_008,295
DEV_003,276
DEV_007,271


All listed devices have experienced critical conditions more than five times.


## Q24. Find the factory with the highest number of abnormal sensor readings.

**Answer:**

In [25]:
factory_abnormal = df.groupby("Location").agg(
    Abnormal_Temperature=("Temp_Abnormal","sum"),
    Abnormal_Vibration=("Vibration_Abnormal","sum")
)
factory_abnormal["Total_Abnormal_Sensor_Readings"] = factory_abnormal["Abnormal_Temperature"] + factory_abnormal["Abnormal_Vibration"]
factory_abnormal = factory_abnormal.sort_values("Total_Abnormal_Sensor_Readings", ascending=False)
display(factory_abnormal)
print("Factory with the highest number of abnormal sensor readings:", factory_abnormal.index[0])

,Abnormal_Temperature,Abnormal_Vibration,Total_Abnormal_Sensor_Readings
Location,,,
Factory_A,66,61,127
Factory_B,63,61,124
Factory_C,67,47,114


Factory with the highest number of abnormal sensor readings: Factory_A


## Q25. Calculate the percentage of time each device operates under warning/critical conditions.

**Answer:**

In [26]:
warning_critical_pct = (
    df.groupby("Device_ID")["Machine_Health"]
      .apply(lambda s: s.isin(["Warning","Critical"]).mean() * 100)
      .sort_values(ascending=False)
      .round(2)
)
display(warning_critical_pct.to_frame("Warning_or_Critical_Percentage"))

,Warning_or_Critical_Percentage
Device_ID,
DEV_006,47.17
DEV_002,46.64
DEV_004,46.60
DEV_005,46.60
DEV_007,45.58
DEV_008,45.52
DEV_001,45.36
DEV_003,45.30


## Q26. Identify the device that requires the highest maintenance priority.

**Answer:**

In [27]:
maintenance_summary = df.groupby("Device_ID").agg(
    Critical_Conditions=("Machine_Health", lambda s: (s == "Critical").sum()),
    Warning_or_Critical_Pct=("Machine_Health", lambda s: s.isin(["Warning","Critical"]).mean() * 100),
    Abnormal_Temperature=("Temp_Abnormal","sum"),
    Abnormal_Vibration=("Vibration_Abnormal","sum"),
    Minimum_Battery=("Battery_Level","min")
)
maintenance_summary["Abnormal_Sensor_Readings"] = (
    maintenance_summary["Abnormal_Temperature"] + maintenance_summary["Abnormal_Vibration"]
)
maintenance_summary = maintenance_summary.sort_values(
    ["Critical_Conditions","Warning_or_Critical_Pct","Minimum_Battery"],
    ascending=[False,False,True]
).round(2)

display(maintenance_summary)
priority_device = maintenance_summary.index[0]
print(f"Highest maintenance priority: {priority_device}")

,Critical_Conditions,Warning_or_Critical_Pct,Abnormal_Temperature,Abnormal_Vibration,Minimum_Battery,Abnormal_Sensor_Readings
Device_ID,,,,,,
DEV_004,323,46.60,27,28,8.51,55
DEV_005,318,46.60,28,17,2.96,45
DEV_002,313,46.64,16,28,2.15,44
DEV_006,306,47.17,18,26,2.20,44
DEV_001,304,45.36,28,28,2.26,56
DEV_008,295,45.52,26,12,2.01,38
DEV_003,276,45.30,29,14,3.67,43
DEV_007,271,45.58,24,16,2.15,40


Highest maintenance priority: DEV_004
